In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/garv3068/results/artifacts/exp4_specialized_metadata.json
/kaggle/input/datasets/garv3068/results/artifacts/baseline_catboost.cbm
/kaggle/input/datasets/garv3068/results/artifacts/baseline_lightgbm.txt
/kaggle/input/datasets/garv3068/results/artifacts/baseline_xgboost.json
/kaggle/input/datasets/garv3068/results/artifacts/lgb_specialized_service.txt
/kaggle/input/datasets/garv3068/results/artifacts/lgb_optimized.txt
/kaggle/input/datasets/garv3068/results/artifacts/submissions/submission_blend_lgb57_xgb24_cat19.csv
/kaggle/input/datasets/garv3068/results/artifacts/oof/exp_diversity_lgb01_blend.npy
/kaggle/input/datasets/garv3068/results/artifacts/oof/baseline_cat.npy
/kaggle/input/datasets/garv3068/results/artifacts/oof/baseline_lgb.npy
/kaggle/input/datasets/garv3068/results/artifacts/oof/baseline_xgb.npy
/kaggle/input/datasets/garv3068/results/artifacts/oof/lgb_optimized.npy
/kaggle/input/datasets/garv3068/results/artifacts/oof/lgb_specialized_service.npy
/kagg

## EXP 3-1 → TabPFN baseline

In [2]:
import os
import numpy as np
import pandas as pd

TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e10/train.csv"
TEST_PATH = "/kaggle/input/competitions/playground-series-s6e10/test.csv"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

print("Train:", train.shape)
print("Test :", test.shape)
print()
print("Target distribution:")
print(train["satisfaction"].value_counts())
print()
print("Columns:")
print(train.columns.tolist())

Train: (699635, 23)
Test : (299844, 22)

Target distribution:
satisfaction
False    389296
True     310339
Name: count, dtype: int64

Columns:
['id', 'Age', 'Flight Distance', 'Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking', 'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort', 'Inflight entertainment', 'On-board service', 'Leg room service', 'Baggage handling', 'Checkin service', 'Cleanliness', 'Departure Delay in Minutes', 'Arrival Delay in Minutes', 'Gender', 'Customer Type', 'Type of Travel', 'Class', 'satisfaction']


In [3]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [4]:
!pip install -q tabpfn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 595.9/595.9 kB 16.7 MB/s eta 0:00:00


In [5]:
import tabpfn
print("TabPFN version:", tabpfn.__version__)

TabPFN version: 9.1.0


## Step 1 — Create the exact validation split

In [6]:
import pandas as pd
# train = pd.read_csv("")
train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e10/train.csv")
test = pd.read_csv("/kaggle/input/competitions/playground-series-s6e10/test.csv")

In [7]:
from sklearn.model_selection import train_test_split

TARGET = "satisfaction"

X = train.drop(columns=[TARGET, "id"])
y = train[TARGET].astype(int)

X_train_tab, X_valid_tab, y_train_tab, y_valid_tab = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Train:", X_train_tab.shape)
print("Valid:", X_valid_tab.shape)
print("Target train:", y_train_tab.mean())
print("Target valid:", y_valid_tab.mean())

Train: (559708, 21)
Valid: (139927, 21)
Target train: 0.44357236273199596
Target valid: 0.4435741493778899


In [8]:
from tabpfn import TabPFNClassifier
import inspect

print(inspect.signature(TabPFNClassifier))

(*, n_estimators: "int | Literal['auto']" = 'auto', auto_scale_n_estimators: 'bool' = True, categorical_features_indices: 'Sequence[int] | None' = None, softmax_temperature: "float | Literal['auto']" = 'auto', balance_probabilities: 'bool' = False, average_before_softmax: 'bool' = False, model_path: "str | Path | list[str] | list[Path] | Literal['auto'] | ModelSpecs | list[ModelSpecs]" = 'auto', device: 'DevicesSpecification' = 'auto', ignore_pretraining_limits: 'bool' = False, inference_precision: "_dtype | Literal['autocast', 'auto']" = 'auto', fit_mode: "Literal['low_memory', 'fit_preprocessors', 'fit_with_cache', 'batched']" = 'fit_preprocessors', memory_saving_mode: 'MemorySavingMode' = 'auto', keep_cache_on_device: 'bool' = True, kv_cache_precision: "Literal['auto', 'int8', 'fp8', 'adaptive'] | None" = None, random_state: 'int | np.random.RandomState | np.random.Generator | None' = 0, n_jobs: "Annotated[int | None, deprecated('Use n_preprocessing_jobs')]" = None, n_preprocessing_

In [9]:
categorical_cols = [
    "Gender",
    "Customer Type",
    "Type of Travel",
    "Class"
]

feature_cols = [
    c for c in train.columns
    if c not in ["id", "satisfaction"]
]

print("Features:", len(feature_cols))
print(feature_cols)

cat_indices = [
    feature_cols.index(c)
    for c in categorical_cols
]

print("\nCategorical indices:", cat_indices)

Features: 21
['Age', 'Flight Distance', 'Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking', 'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort', 'Inflight entertainment', 'On-board service', 'Leg room service', 'Baggage handling', 'Checkin service', 'Cleanliness', 'Departure Delay in Minutes', 'Arrival Delay in Minutes', 'Gender', 'Customer Type', 'Type of Travel', 'Class']

Categorical indices: [17, 18, 19, 20]


In [10]:
X_tab = train[feature_cols].copy()

# TabPFN handles missing values, but make categorical columns explicit
for col in categorical_cols:
    X_tab[col] = X_tab[col].astype("category")

print(X_tab.dtypes)

Age                                     int64
Flight Distance                         int64
Inflight wifi service                   int64
Departure/Arrival time convenient       int64
Ease of Online booking                  int64
Gate location                           int64
Food and drink                          int64
Online boarding                         int64
Seat comfort                            int64
Inflight entertainment                  int64
On-board service                        int64
Leg room service                        int64
Baggage handling                        int64
Checkin service                         int64
Cleanliness                             int64
Departure Delay in Minutes              int64
Arrival Delay in Minutes              float64
Gender                               category
Customer Type                        category
Type of Travel                       category
Class                                category
dtype: object


## EXP 3-1 — TabPFN baseline

In [11]:
from sklearn.model_selection import train_test_split

X_train_full = train.loc[X_train_tab.index, feature_cols].copy()
X_valid_full = train.loc[X_valid_tab.index, feature_cols].copy()

y_train_full = y_train_tab.copy()
y_valid_full = y_valid_tab.copy()

# Controlled TabPFN training subset
X_tab_train, _, y_tab_train, _ = train_test_split(
    X_train_full,
    y_train_full,
    train_size=20_000,
    stratify=y_train_full,
    random_state=2026
)

print("TabPFN train:", X_tab_train.shape)
print("Validation  :", X_valid_full.shape)
print("Train target:", y_tab_train.mean())
print("Valid target:", y_valid_full.mean())

TabPFN train: (20000, 21)
Validation  : (139927, 21)
Train target: 0.44355
Valid target: 0.4435741493778899


In [12]:
from tabpfn import TabPFNClassifier

tabpfn_model = TabPFNClassifier(
    n_estimators=4,
    categorical_features_indices=cat_indices,
    device="cuda",
    fit_mode="batched",
    memory_saving_mode="auto",
    inference_precision="auto",
    random_state=2026,
    show_progress_bar=True
)

print("TabPFN model ready.")

TabPFN model ready.


In [13]:
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

os.environ["TABPFN_TOKEN"] = user_secrets.get_secret("TABPFN_TOKEN")

print("TabPFN token loaded:", bool(os.environ.get("TABPFN_TOKEN")))

TabPFN token loaded: True


In [14]:
import os

token = os.environ.get("TABPFN_TOKEN")

print("Token exists:", token is not None)
print("Token length:", len(token) if token else 0)
print("Token preview:", token[:6] + "..." if token else "NONE")

Token exists: True
Token length: 53
Token preview: tabpfn...


In [15]:
import os

print("TABPFN_TOKEN set:", bool(os.environ.get("TABPFN_TOKEN")))

from tabpfn import TabPFNClassifier

test_auth_model = TabPFNClassifier(
    n_estimators=1,
    device="cuda",
    random_state=2026
)

print("Model object created successfully.")

TABPFN_TOKEN set: True
Model object created successfully.


In [16]:
from tabpfn import TabPFNClassifier

tabpfn_model = TabPFNClassifier(
    n_estimators=4,
    categorical_features_indices=cat_indices,
    device="cuda",
    fit_mode="fit_preprocessors",
    memory_saving_mode="auto",
    inference_precision="auto",
    random_state=2026,
    show_progress_bar=True,
    inference_config={
        "SUBSAMPLE_SAMPLES": 50_000
    }
)

print("TabPFN configured.")

TabPFN configured.


In [17]:
from tabpfn.browser_auth import _get_license_name

# Find the model repo that TabPFN 9.1.0 is using
import tabpfn.model_loading as ml

print("model_loading attributes containing 'REPO' or 'VERSION':")
print([
    x for x in dir(ml)
    if "REPO" in x.upper() or "VERSION" in x.upper()
])

# Also inspect the source used by your installed package
import inspect
from tabpfn import browser_auth

print("\nLicense function source:")
print(inspect.getsource(browser_auth._get_license_name))

model_loading attributes containing 'REPO' or 'VERSION':
['ModelVersion', 'TabPFNHuggingFaceGatedRepoError', '_resolve_model_version', '_version_has_direct_download_option', 'resolve_model_version']

License function source:
def _get_license_name(hf_repo_id: str) -> str:
    """Fetch the license_name from the HuggingFace API for a Prior-Labs repo.

    Uses the user's cached HuggingFace token (``HF_TOKEN`` env var or
    ``huggingface-cli login``) so gated repos resolve correctly. Raises
    :class:`TabPFNLicenseError` if the user has no HF access to the repo,
    the request fails, or the model card has no ``license_name``.
    """
    from huggingface_hub import get_token  # noqa: PLC0415

    url = f"https://huggingface.co/api/models/Prior-Labs/{hf_repo_id}"
    headers = {}
    token = get_token()
    if token:
        headers["Authorization"] = f"Bearer {token}"
    req = urllib.request.Request(url, headers=headers)
    try:
        with urllib.request.urlopen(req, timeout=10) as 

In [18]:
from tabpfn import browser_auth
from tabpfn.model_loading import resolve_model_version

# What model does TabPFN resolve to?
resolved = resolve_model_version("auto")

print("Resolved model version:")
print(resolved)

# Extract the HF repo ID if available
print("\nResolved object type:")
print(type(resolved))

print("\nResolved object:")
print(resolved)

Resolved model version:
ModelVersion.V2

Resolved object type:
<enum 'ModelVersion'>

Resolved object:
ModelVersion.V2


In [19]:
# Get the exact license identifier TabPFN itself will request
hf_repo_id = getattr(resolved, "hf_repo_id", None)

print("HF repo ID:", hf_repo_id)

if hf_repo_id:
    license_name = browser_auth._get_license_name(hf_repo_id)
    print("EXACT LICENSE NAME:", repr(license_name))

HF repo ID: None


In [20]:
from tabpfn import TabPFNClassifier
from tabpfn.constants import ModelVersion

tabpfn_v2 = TabPFNClassifier.create_default_for_version(
    ModelVersion.V2
)

print("TabPFN V2 model created successfully")

TabPFN V2 model created successfully


In [21]:
tabpfn_v2.fit(
    X_train_tab.iloc[:100],
    y_train_tab.iloc[:100]
)

print("V2 FIT SUCCESS")

tabpfn-v2-classifier-finetuned-zk73skhh.(…): reconstructing file:   0%|          |  0.00B / 29.0MB            

tabpfn-v2-classifier-finetuned-zk73skhh.(…): downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/37.0 [00:00<?, ?B/s]

V2 FIT SUCCESS


## Experiment 3A — TabPFN V2 on 20k samples

In [22]:
from tabpfn import TabPFNClassifier
from tabpfn.constants import ModelVersion
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

# ------------------------------------------------------------
# TABPFN V2 — 10K SUPPORTED EXPERIMENT
# ------------------------------------------------------------

X_train_full = train.loc[X_train_tab.index, feature_cols].copy()
X_valid_full = train.loc[X_valid_tab.index, feature_cols].copy()

y_train_full = y_train_tab.copy()
y_valid_full = y_valid_tab.copy()

# 10K stratified training subset
X_tab_10k, _, y_tab_10k, _ = train_test_split(
    X_train_full,
    y_train_full,
    train_size=10_000,
    stratify=y_train_full,
    random_state=2026
)

print("Training shape:", X_tab_10k.shape)

# ------------------------------------------------------------
# MODEL
# ------------------------------------------------------------

tabpfn_v2 = TabPFNClassifier.create_default_for_version(
    ModelVersion.V2
)

tabpfn_v2.device = "cuda"

# ------------------------------------------------------------
# FIT
# ------------------------------------------------------------

tabpfn_v2.fit(
    X_tab_10k,
    y_tab_10k
)

# ------------------------------------------------------------
# VALIDATION
# ------------------------------------------------------------

tab_pred_10k = tabpfn_v2.predict_proba(
    X_valid_full
)[:, 1]

tab_auc = roc_auc_score(
    y_valid_full,
    tab_pred_10k
)

print("=" * 60)
print("TABPFN V2 — 10K")
print("=" * 60)
print(f"AUC:      {tab_auc:.9f}")
print(f"Champion: 0.958378205")
print(f"Gap:      {tab_auc - 0.958378205:+.9f}")

Training shape: (10000, 21)
TABPFN V2 — 10K
AUC:      0.953251283
Champion: 0.958378205
Gap:      -0.005126922


In [23]:
import numpy as np
from sklearn.metrics import roc_auc_score

# ------------------------------------------------------------
# LOAD SAVED OOF PREDICTIONS
# ------------------------------------------------------------

lgb_v = np.load(
    "/kaggle/input/datasets/garv3068/results/artifacts/oof/lgb_optimized.npy"
)

xgb_v = np.load(
    "/kaggle/input/datasets/garv3068/results/artifacts/oof/baseline_xgb.npy"
)

cat_v = np.load(
    "/kaggle/input/datasets/garv3068/results/artifacts/oof/baseline_cat.npy"
)

spec_v = np.load(
    "/kaggle/input/datasets/garv3068/results/artifacts/oof/lgb_specialized_service.npy"
)

# ------------------------------------------------------------
# RECONSTRUCT CURRENT CHAMPION
# ------------------------------------------------------------

old_champion_valid = (
    0.57 * lgb_v
    + 0.24 * xgb_v
    + 0.19 * cat_v
)

champion_valid = (
    0.75 * old_champion_valid
    + 0.25 * spec_v
)

champion_auc = roc_auc_score(
    y_valid_tab,
    champion_valid
)

print("Champion AUC:", champion_auc)
print("Expected:     0.958376553")
print("Champion shape:", champion_valid.shape)
print("TabPFN shape:   ", tab_pred_10k.shape)

# ------------------------------------------------------------
# CORRELATION
# ------------------------------------------------------------

corr = np.corrcoef(
    champion_valid,
    tab_pred_10k
)[0, 1]

print("\nTabPFN ↔ Champion correlation:", corr)

# ------------------------------------------------------------
# BLEND SEARCH
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("TABPFN V2 — BLEND SEARCH")
print("=" * 60)

best_auc = champion_auc
best_w = 0.0

for w in np.arange(0.00, 0.51, 0.02):

    blended = (
        (1 - w) * champion_valid
        + w * tab_pred_10k
    )

    auc = roc_auc_score(
        y_valid_tab,
        blended
    )

    print(
        f"TabPFN={w:.2f} | "
        f"Champion={1-w:.2f} | "
        f"AUC={auc:.9f}"
    )

    if auc > best_auc:
        best_auc = auc
        best_w = w

print("\n" + "=" * 60)
print(f"BEST AUC:       {best_auc:.9f}")
print(f"TABPFN WEIGHT:  {best_w:.2f}")
print(f"IMPROVEMENT:    {best_auc - champion_auc:+.9f}")
print("=" * 60)

Champion AUC: 0.9583765532474063
Expected:     0.958376553
Champion shape: (139927,)
TabPFN shape:    (139927,)

TabPFN ↔ Champion correlation: 0.9890145565979228

TABPFN V2 — BLEND SEARCH
TabPFN=0.00 | Champion=1.00 | AUC=0.958376553
TabPFN=0.02 | Champion=0.98 | AUC=0.958368738
TabPFN=0.04 | Champion=0.96 | AUC=0.958352863
TabPFN=0.06 | Champion=0.94 | AUC=0.958331709
TabPFN=0.08 | Champion=0.92 | AUC=0.958305769
TabPFN=0.10 | Champion=0.90 | AUC=0.958276469
TabPFN=0.12 | Champion=0.88 | AUC=0.958245219
TabPFN=0.14 | Champion=0.86 | AUC=0.958211643
TabPFN=0.16 | Champion=0.84 | AUC=0.958175142
TabPFN=0.18 | Champion=0.82 | AUC=0.958135533
TabPFN=0.20 | Champion=0.80 | AUC=0.958092836
TabPFN=0.22 | Champion=0.78 | AUC=0.958047778
TabPFN=0.24 | Champion=0.76 | AUC=0.957999480
TabPFN=0.26 | Champion=0.74 | AUC=0.957947705
TabPFN=0.28 | Champion=0.72 | AUC=0.957892250
TabPFN=0.30 | Champion=0.70 | AUC=0.957834153
TabPFN=0.32 | Champion=0.68 | AUC=0.957771992
TabPFN=0.34 | Champion=0.66 |

In [24]:
import numpy as np
from sklearn.metrics import roc_auc_score

# Current champion validation prediction
champion_valid = (
    0.75 * (
        0.57 * lgb_v
        + 0.24 * xgb_v
        + 0.19 * cat_v
    )
    + 0.25 * spec_v
)

# TabPFN correlation
corr = np.corrcoef(
    champion_valid,
    tab_pred_10k
)[0, 1]

print("TabPFN ↔ Champion correlation:", corr)

# ------------------------------------------------------------
# BLEND SEARCH
# ------------------------------------------------------------

print("\nTabPFN blend search")
print("=" * 60)

best_auc = 0
best_w = None

for w in np.arange(0.00, 0.51, 0.02):
    blended = (
        (1 - w) * champion_valid
        + w * tab_pred_10k
    )

    auc = roc_auc_score(y_valid_tab, blended)

    print(
        f"TabPFN={w:.2f} | "
        f"Champion={1-w:.2f} | "
        f"AUC={auc:.9f}"
    )

    if auc > best_auc:
        best_auc = auc
        best_w = w

print("\n" + "=" * 60)
print(f"BEST AUC: {best_auc:.9f}")
print(f"BEST TABPFN WEIGHT: {best_w:.2f}")
print(f"IMPROVEMENT: {best_auc - 0.958378205:+.9f}")

TabPFN ↔ Champion correlation: 0.9890145565979228

TabPFN blend search
TabPFN=0.00 | Champion=1.00 | AUC=0.958376553
TabPFN=0.02 | Champion=0.98 | AUC=0.958368738
TabPFN=0.04 | Champion=0.96 | AUC=0.958352863
TabPFN=0.06 | Champion=0.94 | AUC=0.958331709
TabPFN=0.08 | Champion=0.92 | AUC=0.958305769
TabPFN=0.10 | Champion=0.90 | AUC=0.958276469
TabPFN=0.12 | Champion=0.88 | AUC=0.958245219
TabPFN=0.14 | Champion=0.86 | AUC=0.958211643
TabPFN=0.16 | Champion=0.84 | AUC=0.958175142
TabPFN=0.18 | Champion=0.82 | AUC=0.958135533
TabPFN=0.20 | Champion=0.80 | AUC=0.958092836
TabPFN=0.22 | Champion=0.78 | AUC=0.958047778
TabPFN=0.24 | Champion=0.76 | AUC=0.957999480
TabPFN=0.26 | Champion=0.74 | AUC=0.957947705
TabPFN=0.28 | Champion=0.72 | AUC=0.957892250
TabPFN=0.30 | Champion=0.70 | AUC=0.957834153
TabPFN=0.32 | Champion=0.68 | AUC=0.957771992
TabPFN=0.34 | Champion=0.66 | AUC=0.957706629
TabPFN=0.36 | Champion=0.64 | AUC=0.957638154
TabPFN=0.38 | Champion=0.62 | AUC=0.957566560
TabPFN=0.